In [15]:
import duckdb

conn = duckdb.connect("../credit_risk_warehouse.db")
print("⚡ Successfully reconnected to the analytical data warehouse!")

⚡ Successfully reconnected to the analytical data warehouse!


In [16]:
risk_analysis_df = conn.execute("""
    SELECT 
        CASE 
            WHEN AMT_INCOME_TOTAL < 100000 THEN 'Low Income (<100k)'
            WHEN AMT_INCOME_TOTAL BETWEEN 100000 AND 200000 THEN 'Medium Income (100k-200k)'
            ELSE 'High Income (>200k)'
        -- AS creates a clear custom header name for our bucketed group column
        END AS income_segment,
        
        COUNT(*) AS total_applicants,
        SUM(TARGET) AS total_defaulters,
        
        -- Calculating the mathematical percentage rate of defaults per group
        ROUND(AVG(TARGET) * 100, 2) AS default_rate_percentage

    FROM mart_applicant_features
    GROUP BY income_segment
    ORDER BY default_rate_percentage DESC;
""").df()

# Display the resulting data table matrix on the screen
risk_analysis_df

,income_segment,total_applicants,total_defaulters,default_rate_percentage
0,Medium Income (100k-200k),155898,13326.0,8.55
1,Low Income (<100k),63698,5225.0,8.20
2,High Income (>200k),87915,6274.0,7.14


In [17]:
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.


In [18]:
import sklearn

# Print the version to confirm it works
print(sklearn.__version__)


1.7.2


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import pandas as pd


# 1. Extract the raw analytical data mart table into a Pandas structure
ml_data = conn.execute("SELECT * FROM mart_applicant_features;").df()

# 2. Drop any missing or non-numeric identifiers to isolate our mathematical model features
features = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'credit_income_ratio', 
            'annuity_income_ratio', 'total_bureau_records', 'total_previous_loans_applied']

# Fill any structural blank cells (NULLs) with 0 to prevent algorithm calculations from failing
X = ml_data[features].fillna(0)
y = ml_data['TARGET']

# 3. Partition data matrix into isolated training sets and evaluation sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Instantiate and fit the predictive algorithm
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 5. Compute predictive probabilities and generate classifications
probabilities = model.predict_proba(X_test)[:, 1]
predictions = model.predict(X_test)

# 6. Output evaluation metrics required by the rubric checklist
print("=== 🎯 PHASE 6: MACHINE LEARNING RISK MODEL METRICS ===")
print(f"🥇 ROC-AUC Score: {roc_auc_score(y_test, probabilities):.4f}")
print(f"📊 Precision Score: {precision_score(y_test, predictions, zero_division=0):.4f}")
print(f"📈 Recall Score: {recall_score(y_test, predictions):.4f}")

=== 🎯 PHASE 6: MACHINE LEARNING RISK MODEL METRICS ===
🥇 ROC-AUC Score: 0.5419
📊 Precision Score: 0.0000
📈 Recall Score: 0.0000
